# E07_RacingCar — подбор гиперпараметров

**Урок:** 3.6 «Оптимизация гиперпараметров: Optuna + W&B» курса
[cyber-unity-learn](https://rl-cuber-unity-code.com/courses/3-6)
**Что реализуется:** случайный поиск и последовательное деление пополам
(successive halving) своим кодом, на среде `E07_RacingCar` с SAC.
**Ожидаемый результат:** конфигурация лучше базовой при **меньшем**
суммарном бюджете, чем у полного перебора.
**Время:** зависит от бюджета; при `QUICK_RUN = True` — около 15 минут.

## Почему свой код, а не Optuna

Урок разбирает HPO через Optuna и W&B Sweeps. Здесь те же два базовых
алгоритма написаны своими руками — по той же причине, по которой своими
руками написаны PPO и SAC: **учебный код не должен прятать происходящее
за библиотечным вызовом**. Оба умещаются в полторы сотни строк
(`python/labrl/eval/hpo.py`), и в них нечего скрывать.

Запрет 16.5 на сторонние библиотеки касается реализаций **алгоритмов
обучения**; Optuna под него не подпадает — она не RL-библиотека. Подключить
её к тому же интерфейсу `Trial` можно десятком строк, и это хорошее
упражнение после этого ноутбука.

## Два метода и в чём между ними разница

**Случайный поиск** берёт `n` случайных точек и обучает каждую полным
бюджетом. Незаменим как точка отсчёта: любой более умный метод обязан его
обыгрывать, иначе он не нужен.

Почему случайный, а не по сетке: при `k` параметрах сетка из `m` значений
требует `m^k` прогонов и даёт каждому параметру всего `m` различных значений.
Случайный поиск за те же `n` прогонов даёт `n` различных значений **каждому**
параметру. На практике важны один-два параметра из десяти, а какие именно —
заранее неизвестно.

**Последовательное деление пополам** обучает все точки коротким бюджетом,
отбрасывает худшую половину, оставшимся удваивает бюджет. За тот же
суммарный бюджет проверяет втрое-вчетверо больше конфигураций.

Его известная слабость: медленно стартующая, но сильная в пределе
конфигурация будет отброшена рано. Это свойство метода, а не дефект
реализации, и повод не доверять ему одному.

## Требования

```powershell
python scripts\build_env.py E07_RacingCar
```

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")

## 2. Пространство поиска

Три параметра SAC, и у каждого своя шкала — это не формальность:

| Параметр | Шкала | Почему |
|---|---|---|
| `learning_rate` | **log_uniform** | важно **отношение**: 1e-5 и 1e-4 отличаются вдесятеро, а 1e-3 и 2e-3 — вдвое. Равномерная выборка на отрезке отдала бы почти все точки большим значениям |
| `tau` | log_uniform | то же: скорость мягкого обновления целевой сети меняется на порядки |
| `init_alpha` | uniform | температура энтропии — величина порядка единицы, важна разность |
| `batch_size` | choice | дискретен по природе |

Границы взяты вокруг значений из `configs/E07_RacingCar__sac.yaml`: подбор
должен **уточнять** рабочую конфигурацию, а не искать в пустоте.

In [ ]:
from labrl.eval.hpo import SearchSpace

SPACE = SearchSpace({
    "learning_rate": ("log_uniform", 1e-5, 1e-3),
    "tau":           ("log_uniform", 1e-3, 5e-2),
    "init_alpha":    ("uniform", 0.05, 0.5),
    "batch_size":    ("choice", [128, 256, 512]),
})

print(SPACE.describe())

rng = np.random.default_rng(SEED)
print("\nтри случайные точки:")
for _ in range(3):
    point = SPACE.sample(rng)
    print("  " + ", ".join(f"{k}={v:.5g}" if isinstance(v, float) else f"{k}={v}"
                           for k, v in point.items()))

## 3. Целевая функция

Одна конфигурация — один прогон обучения с заданным бюджетом. Возвращается
**доля успешных заездов**, а не награда: у `E07_RacingCar` награда содержит
и премию за контрольные точки, и штраф за время, и мерой качества
не является (T-17 в `docs/07_TROUBLESHOOTING.md`).

**Сид фиксирован на все конфигурации.** Иначе подбор сравнивал бы удачу
инициализации, а не гиперпараметры: разброс между сидами у SAC сопоставим
с разбросом между разумными конфигурациями.

In [ ]:
QUICK_RUN = True   # True — маленькие бюджеты (~15 мин); False — полные (~3 часа)

from labrl.algos.sac import SAC, SACConfig
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv
from labrl.eval.success import TERMINATED
from labrl.logging.tb_logger import TBLogger
from labrl.nets.mlp import MLPContinuousQNetwork
from labrl.nets.squashed_gaussian import SquashedGaussianPolicy
from labrl.train.sac import SACTrainConfig, train_sac

ENV_ID = "E07_RacingCar"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED,
                        time_scale=20.0, no_graphics=True, num_areas=8)
vec = VecUnityEnv(handle)
vec.reset()

OBS_DIM = vec.single_obs_dim
ACTION_DIM = int(vec.action_spec.continuous_size)
print(f"obs_dim: {OBS_DIM}, непрерывных действий: {ACTION_DIM}, арен: {vec.num_envs}")


def objective(params, budget):
    """Обучает SAC с заданными гиперпараметрами и возвращает долю успехов.

    Сид фиксирован: подбор обязан сравнивать конфигурации, а не удачу
    инициализации.
    """
    set_global_seed(SEED)

    policy = SquashedGaussianPolicy(OBS_DIM, ACTION_DIM, (256, 256), "relu")
    q1 = MLPContinuousQNetwork(OBS_DIM, ACTION_DIM, (256, 256), "relu")
    q2 = MLPContinuousQNetwork(OBS_DIM, ACTION_DIM, (256, 256), "relu")

    algo = SAC(
        policy, q1, q2,
        SACConfig(
            gamma=0.99,
            learning_rate=float(params["learning_rate"]),
            batch_size=int(params["batch_size"]),
            tau=float(params["tau"]),
            init_alpha=float(params["init_alpha"]),
            autotune_alpha=True,
        ),
        device=DEVICE, seed=SEED,
    )

    # Логгер в никуда: сотня прогонов подбора не должна засорять results/.
    # Интересен один итог, а не кривые каждой конфигурации.
    logger = TBLogger(None)
    result = train_sac(
        vec=vec, algo=algo, logger=logger,
        cfg=SACTrainConfig(
            total_steps=int(budget),
            buffer_size=100_000,
            learning_starts=1_000,
            eval_every_steps=max(1, int(budget)),   # оценка один раз, в конце
            eval_episodes=10,
            success_rule=TERMINATED,
        ),
        seed=SEED,
    )
    logger.close()
    return float(result.last_eval_success)


print("целевая функция готова")

## 4. Случайный поиск — точка отсчёта

Каждая конфигурация обучается **полным** бюджетом. Это дорого и именно
поэтому служит эталоном: любой более умный метод обязан обыграть его
при равном или меньшем суммарном бюджете.

In [ ]:
from labrl.eval.hpo import random_search

RANDOM_TRIALS = 6 if QUICK_RUN else 16
FULL_BUDGET = 8_000 if QUICK_RUN else 60_000

random_result = random_search(
    SPACE, objective, trials=RANDOM_TRIALS, budget=FULL_BUDGET, seed=SEED,
    on_trial=lambda i, t: print(f"  [{i + 1}/{RANDOM_TRIALS}] {t.describe()}"),
)

print()
print(random_result.describe(count=3))

## 5. Последовательное деление пополам

Все конфигурации стартуют коротким бюджетом; после каждого этапа выживает
лучшая половина, а бюджет удваивается. Суммарный бюджет каждого этапа
примерно одинаков — в этом весь приём: ресурс перераспределяется
от заведомо слабых конфигураций к тем, что подают надежды.

In [ ]:
from labrl.eval.hpo import successive_halving

HALVING_TRIALS = 8 if QUICK_RUN else 32
MIN_BUDGET = 2_000 if QUICK_RUN else 8_000

halving_result = successive_halving(
    SPACE, objective, trials=HALVING_TRIALS, min_budget=MIN_BUDGET,
    reduction=2, seed=SEED,
    on_trial=lambda stage, t: print(f"  этап {stage}: {t.describe()}"),
)

print()
print(halving_result.describe(count=3))

## 6. Сравнение

Сравнивать методы надо по **двум** числам сразу: какое качество найдено
и сколько шагов среды на это потрачено. Метод, нашедший чуть лучшую
конфигурацию вдесятеро дороже, задачу не решает.

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

rows = [
    ("случайный поиск", random_result),
    ("деление пополам", halving_result),
]

print(f"{'метод':<20} {'конфигураций':>13} {'бюджет, шагов':>15} {'лучшая доля успехов':>21}")
print("-" * 72)
for name, result in rows:
    print(f"{name:<20} {len(result.trials):>13} {result.total_budget:>15} "
          f"{result.best.score:>20.0%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].bar([name for name, _ in rows], [r.total_budget for _, r in rows],
            color=["tab:blue", "tab:green"])
axes[0].set_title("Суммарный бюджет, шагов среды")
axes[0].grid(alpha=0.3, axis="y")

axes[1].bar([name for name, _ in rows], [r.best.score for _, r in rows],
            color=["tab:blue", "tab:green"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Лучшая найденная доля успехов")
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

better = min(rows, key=lambda pair: pair[1].total_budget / max(pair[1].best.score, 1e-6))
print(f"\nлучшее отношение «бюджет / качество»: {better[0]}")

### Как читать этот результат

**Один прогон подбора — не вывод.** Обе схемы стохастичны, и при другом сиде
порядок может измениться. Требование 12.3 инструкции о сравнении по нескольким
сидам относится и к подбору: чтобы утверждать «деление пополам лучше»,
нужно повторить оба метода на нескольких сидах и сравнить распределения.

**Найденная конфигурация — гипотеза, а не результат.** Она измерена на одном
сиде и коротком бюджете. Прежде чем менять `configs/E07_RacingCar__sac.yaml`,
её нужно проверить полным прогоном на трёх сидах — так же, как любую другую.

## 7. Что делать с найденной конфигурацией

In [ ]:
best = min(rows, key=lambda pair: pair[1].total_budget / max(pair[1].best.score, 1e-6))[1].best

print("лучшая найденная конфигурация:")
for key, value in best.params.items():
    formatted = f"{value:.6g}" if isinstance(value, float) else value
    print(f"  {key}: {formatted}")

print("\nчтобы проверить её полным прогоном, впишите эти значения в блок `algo`")
print("файла configs/E07_RacingCar__sac.yaml и запустите:")
print("  python scripts/train.py --config configs/E07_RacingCar__sac.yaml --all-seeds")

vec.handle.close()

## 8. Что здесь не реализовано

**Байесовская оптимизация и TPE** (тоже из урока 3.6). Оба требуют
суррогатной модели пространства и правила выбора следующей точки —
это отдельная работа, и без неё две базовые схемы уже дают рабочий
инструмент.

**Hyperband** — надстройка над делением пополам, перебирающая разные
стартовые бюджеты, чтобы застраховаться от ранней отбраковки медленно
стартующих конфигураций. Реализуется поверх `successive_halving`
несколькими строками и оставлен как упражнение.

**W&B Sweeps** — инфраструктура запуска, а не алгоритм; в лаборатории
её роль выполняет `scripts/train.py --worker-id`, позволяющий вести
несколько прогонов параллельно на разных портах.